# Memória de Cálculo — Capacidade Portuária (Carga Conteinerizada)

**Roteiro Metodológico para Análise de Capacidades em Terminais Portuários**  
LabPortos/UFMA — Infra S.A.

Este notebook implementa os Passos 1 e 2 do roteiro metodológico **exclusivamente para carga conteinerizada** (TEU), gerando:

- **Planilha 1** — Base depurada com tratamento de replicatas e tempos operacionais calculados (`Planilha1_Base_Depurada_Cont.xlsx`)
- **Planilha 2** — Indicadores operacionais por grupo, após depuração por IQR (`Planilha2_Indicadores_Operacionais_Cont.xlsx`)
- **Planilha 3** — Capacidades calculadas com BOR observado e BUR por berço (`Planilha3_Capacidades_Cont.xlsx`)

**Diferença em relação à base de cargas não conteinerizadas:**  
A ANTAQ registra a mesma atracação em múltiplas linhas na base de contêineres. O tratamento de replicatas é obrigatório antes de qualquer cálculo: agrupa por IMO + cinco *timestamps* e soma a movimentação em TEU, mantendo um único registro por atracação.

---
**Parâmetros ajustáveis** — revise a célula de configuração antes de executar.


In [ ]:
# ============================================================
# 0. BIBLIOTECAS
# ============================================================
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Bibliotecas carregadas.')

In [ ]:
# ============================================================
# 1. PARÂMETROS DE ENTRADA
# ============================================================

# --- Arquivo de entrada ---
ARQUIVO_ENTRADA = 'BD_Bruta_ANTAQ_Cont.xlsx'
SHEET_CONT     = 'Sheet1'      # aba com os dados na base de contêineres da ANTAQ

# --- Parâmetros da Eq. 1b (Roteiro §3.8) ---
CLEARANCE_H = 3.0     # intervalo entre atracações (h)
H_CAL       = 8760    # horas do calendário anual

# H_ef calculado pela Eq. 1c: H_cal − H_cli − H_mnt − H_nav − H_out
# Substitua pelo valor do porto em análise.
H_EF = 8000

# ─────────────────────────────────────────────────────────────────────────
# BOR_adm  (Quadro 17 do Roteiro Metodológico — UNCTAD/PIANC)
# ─────────────────────────────────────────────────────────────────────────
# NOTA DE DIVERGÊNCIA EM RELAÇÃO AO CÓDIGO DE PARANAGUÁ
# O código original dos Planos Mestres (Paranaguá/Maceió) usou:
#     365 * 24 * 0.80 / Tempo_por_Navio  →  BOR_adm = 0,80 fixo, H_ef = 8.760 h
# para TODOS os perfis, incluindo contêineres.
#
# Para carga conteinerizada, o Quadro 17 do roteiro (UNCTAD, 1985) recomenda
# limites mais conservadores: 0,50 para 1 berço, 0,65 para 2–3 berços e
# 0,70 para ≥ 4 berços. Isso reflete a maior variabilidade de Ta em
# terminais de contêineres em relação a terminais de granel.
#
# Este notebook usa H_EF (horas operacionais reais, não 8.760) e
# BOR_adm diferenciado por número de berços — maior generalização
# em relação ao código de Paranaguá, alinhada ao Quadro 17 do roteiro.
# ─────────────────────────────────────────────────────────────────────────

# BOR_adm por número de berços — contêineres (Quadro 17, UNCTAD 1985)
BOR_ADM_CONT_MAPA = {
    1:  0.50,   # 1 berço  — limite mais restritivo (M/M/1)
    2:  0.65,   # 2–3 berços — efeito de pooling moderado
    3:  0.65,
    99: 0.70,   # ≥ 4 berços — pooling pronunciado
}

def get_bor_adm_cont(n_bercos):
    """Retorna BOR_adm para contêineres conforme número de berços (Quadro 17)."""
    if n_bercos <= 1:   return BOR_ADM_CONT_MAPA[1]
    elif n_bercos <= 3: return BOR_ADM_CONT_MAPA[2]
    else:               return BOR_ADM_CONT_MAPA[99]

# Para ULCSs, adotar 0,60 no lugar de 0,65 (§3.12 do roteiro).
# Ajuste: BOR_ADM_CONT_MAPA[2] = 0.60

# --- Fator de conversão TEU (Quadro 8 do roteiro) ---
FATOR_TEU = 1.55   # TEU por caixa — calibrar com dados locais do terminal

print('Parâmetros configurados.')
print(f'  H_ef = {H_EF} h/ano | Clearance = {CLEARANCE_H} h | Fator TEU = {FATOR_TEU}')
print(f'  BOR_adm: 1 berço={get_bor_adm_cont(1)} | 2–3={get_bor_adm_cont(2)} | 4+={get_bor_adm_cont(99)} (Quadro 17)')


In [ ]:
# ============================================================
# 2. IMPORTAÇÃO DA BASE BRUTA
# Fonte: Estatístico Aquaviário ANTAQ — base de contêineres,
# com movimentação em TEU (Twenty-foot Equivalent Unit).
# ============================================================
bd_bruta = pd.read_excel(ARQUIVO_ENTRADA, sheet_name=SHEET_CONT)

# Filtra apenas registros de carga conteinerizada
bd_bruta = bd_bruta[bd_bruta['Perfil da Carga'] == 'Carga Conteinerizada'].copy()

print(f'Base importada (somente conteinerizada): {len(bd_bruta):,} registros')

# Chave de agrupamento
bd_bruta['Ano'] = bd_bruta['Ano'].astype(str)
bd_bruta['Apoio Concatenar'] = (
    bd_bruta['Ano'] +
    bd_bruta['Nome do Terminal'] +
    bd_bruta['Berço'] +
    bd_bruta['Nomenclatura Simplificada'] +
    bd_bruta['Perfil da Carga'] +
    bd_bruta['Sentido'] +
    bd_bruta['Navegação da Atracação']
)
print(f'Grupos distintos: {bd_bruta["Apoio Concatenar"].nunique():,}')

In [ ]:
# ============================================================
# 3. FUNÇÃO: tratamento_base  →  PLANILHA 1
# Específica para contêineres (Roteiro §2.3 e §11.3):
#   a) Tratamento de replicatas: agrupa por IMO + 5 timestamps
#      e soma movimentação em TEU (a ANTAQ registra a mesma
#      atracação múltiplas vezes nesta base)
#   b) Calcula os cinco tempos operacionais
#   c) Calcula produtividade em TEU/h
#   d) Descarta registros com T_op ≤ 0
# ============================================================
def tratamento_base(df):
    """
    Recebe a base bruta de contêineres e retorna a base depurada
    (Planilha 1) com replicatas tratadas e tempos calculados.
    """
    col_teu = 'Total de Movimentação Contêineres (Bruto)\n(TEU) Twenty Feet Equivalent Unit'
    cols_ts = [
        'IMO da embarcação',
        'Data da Chegada', 'Data da Atracação',
        'Data do Início da Operação', 'Data do Término da Operação',
        'Data da Desatracação'
    ]

    bd = df.copy()
    bd[col_teu] = pd.to_numeric(bd[col_teu], errors='coerce')

    # --- Tratamento de replicatas (Roteiro §2.3) ---
    # Agrupa por IMO + timestamps e soma TEU
    n_antes = len(bd)
    agg_rep = bd.groupby(cols_ts, dropna=False, as_index=False).agg({col_teu: 'sum'})
    bd = bd.merge(agg_rep, on=cols_ts, how='left', suffixes=('_orig', '_soma'))
    bd[col_teu] = bd[col_teu + '_soma']
    bd.drop(columns=[col_teu + '_orig', col_teu + '_soma'], inplace=True)
    bd = bd.drop_duplicates(subset=cols_ts, keep='first')
    n_depois = len(bd)
    print(f'  Replicatas: {n_antes:,} → {n_depois:,} registros únicos '
          f'({n_antes - n_depois:,} replicatas consolidadas)')

    # --- Conversão de datas ---
    col_datas = [
        'Data da Chegada', 'Data da Atracação',
        'Data do Início da Operação', 'Data do Término da Operação',
        'Data da Desatracação'
    ]
    for col in col_datas:
        bd[col] = pd.to_datetime(bd[col], errors='coerce')

    # --- Cálculo dos tempos operacionais ---
    bd['Inop.Pré'] = (
        bd['Data do Início da Operação'] - bd['Data da Atracação']
    ).dt.total_seconds() / 3600

    bd['T_op'] = (
        bd['Data do Término da Operação'] - bd['Data do Início da Operação']
    ).dt.total_seconds() / 3600

    bd['Inop.Pós'] = (
        bd['Data da Desatracação'] - bd['Data do Término da Operação']
    ).dt.total_seconds() / 3600

    bd['Line-up'] = (
        bd['Data da Atracação'] - bd['Data da Chegada']
    ).dt.total_seconds() / 3600

    # --- Produtividade (TEU/h) ---
    for col in ['Inop.Pré', 'T_op', 'Inop.Pós', 'Line-up']:
        bd[col] = pd.to_numeric(bd[col], errors='coerce')

    bd['Produtividade (TEU/h)'] = np.where(
        bd['T_op'] > 0,
        bd[col_teu] / bd['T_op'],
        np.nan
    )

    # --- Descarte de T_op ≤ 0 ---
    n_antes2 = len(bd)
    bd = bd[bd['T_op'] > 0].copy()
    print(f'  Descarte T_op ≤ 0: {n_antes2 - len(bd):,} registros | Resultado: {len(bd):,}')

    return bd


bd_depurada = tratamento_base(bd_bruta)
bd_depurada.head(3)

In [ ]:
# --- Exporta Planilha 1 ---
bd_depurada.to_excel('Planilha1_Base_Depurada_Cont.xlsx', index=False)
print('Planilha 1 exportada: Planilha1_Base_Depurada_Cont.xlsx')

In [ ]:
# ============================================================
# 4. FUNÇÃO: tabela_indicadores_operacionais  →  PLANILHA 2
# Mesma lógica do notebook de não conteinerizados, adaptada
# para TEU:
#   a) IQR em Inop.Pré, Produtividade (TEU/h) e Inop.Pós
#   b) Lote médio em TEU
#   c) Tempo atracado (Ta) = Inop.Pré + T_op + Inop.Pós
#   d) Coluna extra: Lote médio (t) via fator TEU→t
# ============================================================
def tabela_indicadores_operacionais(bd):
    """
    Retorna a tabela de indicadores operacionais por grupo (Planilha 2)
    para carga conteinerizada, com movimentação em TEU.
    """
    col_teu = 'Total de Movimentação Contêineres (Bruto)\n(TEU) Twenty Feet Equivalent Unit'

    cols_id = [
        'Apoio Concatenar', 'Complexo Portuário', 'Nome do Terminal',
        'Berço', 'Ano', 'Perfil da Carga', 'Nomenclatura Simplificada',
        'Sentido', 'Navegação da Atracação'
    ]
    tab = bd[cols_id].drop_duplicates(subset='Apoio Concatenar').copy()

    # --- IQR: Inop.Pré ---
    q1 = bd.groupby('Apoio Concatenar')['Inop.Pré'].quantile(0.25)
    q3 = bd.groupby('Apoio Concatenar')['Inop.Pré'].quantile(0.75)
    tab['Q1_Inop.Pré']    = tab['Apoio Concatenar'].map(q1)
    tab['Q3_Inop.Pré']    = tab['Apoio Concatenar'].map(q3)
    tab['L.Inf_Inop.Pré'] = tab['Q1_Inop.Pré'] - 1.5*(tab['Q3_Inop.Pré']-tab['Q1_Inop.Pré'])
    tab['L.Sup_Inop.Pré'] = tab['Q3_Inop.Pré'] + 1.5*(tab['Q3_Inop.Pré']-tab['Q1_Inop.Pré'])

    # --- IQR: Produtividade (TEU/h) ---
    q1p = bd.groupby('Apoio Concatenar')['Produtividade (TEU/h)'].quantile(0.25)
    q3p = bd.groupby('Apoio Concatenar')['Produtividade (TEU/h)'].quantile(0.75)
    tab['Q1_Produtiv. (TEU/h)']    = tab['Apoio Concatenar'].map(q1p)
    tab['Q3_Produtiv. (TEU/h)']    = tab['Apoio Concatenar'].map(q3p)
    tab['L.Inf_Produtiv.'] = tab['Q1_Produtiv. (TEU/h)'] - 1.5*(tab['Q3_Produtiv. (TEU/h)']-tab['Q1_Produtiv. (TEU/h)'])
    tab['L.Sup_Produtiv.'] = tab['Q3_Produtiv. (TEU/h)'] + 1.5*(tab['Q3_Produtiv. (TEU/h)']-tab['Q1_Produtiv. (TEU/h)'])

    # --- IQR: Inop.Pós ---
    q1s = bd.groupby('Apoio Concatenar')['Inop.Pós'].quantile(0.25)
    q3s = bd.groupby('Apoio Concatenar')['Inop.Pós'].quantile(0.75)
    tab['Q1_Inop.Pós']    = tab['Apoio Concatenar'].map(q1s)
    tab['Q3_Inop.Pós']    = tab['Apoio Concatenar'].map(q3s)
    tab['L.Inf_Inop.Pós'] = tab['Q1_Inop.Pós'] - 1.5*(tab['Q3_Inop.Pós']-tab['Q1_Inop.Pós'])
    tab['L.Sup_Inop.Pós'] = tab['Q3_Inop.Pós'] + 1.5*(tab['Q3_Inop.Pós']-tab['Q1_Inop.Pós'])

    # --- Merge limites para filtro ---
    bd2 = bd.merge(
        tab[['Apoio Concatenar',
             'L.Inf_Inop.Pré','L.Sup_Inop.Pré',
             'L.Inf_Produtiv.','L.Sup_Produtiv.',
             'L.Inf_Inop.Pós','L.Sup_Inop.Pós']],
        on='Apoio Concatenar', how='left'
    )

    def media_iqr(grupo_df, col, col_inf, col_sup):
        mask = (grupo_df[col] >= grupo_df[col_inf]) & (grupo_df[col] <= grupo_df[col_sup])
        return grupo_df.loc[mask, col].mean()

    media_pre  = {}
    media_prod = {}
    media_pos  = {}
    for chave, grp in bd2.groupby('Apoio Concatenar'):
        media_pre[chave]  = media_iqr(grp, 'Inop.Pré',            'L.Inf_Inop.Pré',  'L.Sup_Inop.Pré')
        media_prod[chave] = media_iqr(grp, 'Produtividade (TEU/h)','L.Inf_Produtiv.', 'L.Sup_Produtiv.')
        media_pos[chave]  = media_iqr(grp, 'Inop.Pós',            'L.Inf_Inop.Pós',  'L.Sup_Inop.Pós')

    tab['Média_Inop.Pré (h)']       = tab['Apoio Concatenar'].map(media_pre)
    tab['Média_Produtiv. (TEU/h)']  = tab['Apoio Concatenar'].map(media_prod)
    tab['Média_Inop.Pós (h)']       = tab['Apoio Concatenar'].map(media_pos)

    # --- Agregação de movimentação e atracações ---
    agg = bd.groupby('Apoio Concatenar').agg(
        Lote_medio_TEU = (col_teu, 'mean'),
        Movimentacao_TEU = (col_teu, 'sum'),
        Estadia_media  = ('Line-up', 'mean'),
        Atracacoes     = ('Data da Atracação', 'count'),
    ).reset_index()
    agg.columns = [
        'Apoio Concatenar', 'Lote médio (TEU)', 'Movimentação total (TEU)',
        'Line-up médio (h)', 'N.º atracações'
    ]
    tab = tab.merge(agg, on='Apoio Concatenar', how='left')

    # --- Conversão TEU → toneladas (Quadro 8 do roteiro) ---
    # Lote médio (t) = Lote médio (TEU) × FATOR_TEU
    # Calibrar FATOR_TEU com dados do Estatístico ANTAQ do terminal.
    tab['Lote médio (t) — ref.'] = tab['Lote médio (TEU)'] * FATOR_TEU

    # --- Tempos derivados ---
    tab['T_op recalc. (h)']    = tab['Lote médio (TEU)'] / tab['Média_Produtiv. (TEU/h)']
    tab['Ta — berth time (h)'] = (
        tab['Média_Inop.Pré (h)'] +
        tab['T_op recalc. (h)'] +
        tab['Média_Inop.Pós (h)']
    )
    tab['Clearance a (h)'] = CLEARANCE_H
    tab['Ta + a (h)']      = tab['Ta — berth time (h)'] + CLEARANCE_H

    print(f'  tabela_indicadores: {len(tab):,} grupos')
    return tab


tab_indicadores = tabela_indicadores_operacionais(bd_depurada)
tab_indicadores.head(3)

In [ ]:
# --- Exporta Planilha 2 ---
tab_indicadores.to_excel('Planilha2_Indicadores_Operacionais_Cont.xlsx', index=False)
print('Planilha 2 exportada: Planilha2_Indicadores_Operacionais_Cont.xlsx')

In [ ]:
# ============================================================
# 5. FUNÇÃO: tabela_capacidades_calc  →  PLANILHA 3
# Mesma lógica do notebook de não conteinerizados, com
# capacidade expressa em TEU/ano além de t/ano.
# ============================================================
def tabela_capacidades_calc(tab_ind):
    """
    Retorna a tabela de capacidades calculadas (Planilha 3)
    para carga conteinerizada em TEU/ano e t/ano.
    """
    cols = [
        'Apoio Concatenar', 'Ano', 'Complexo Portuário', 'Nome do Terminal',
        'Berço', 'Perfil da Carga', 'Nomenclatura Simplificada',
        'Sentido', 'Navegação da Atracação',
        'Lote médio (TEU)', 'Lote médio (t) — ref.',
        'Movimentação total (TEU)',
        'N.º atracações',
        'Média_Inop.Pré (h)', 'T_op recalc. (h)', 'Média_Inop.Pós (h)',
        'Ta — berth time (h)', 'Clearance a (h)', 'Ta + a (h)'
    ]
    tab = tab_ind[cols].copy()

    # --- BOR_adm: diferenciado por número de berços (Quadro 17) ---
    # DIFERENÇA EM RELAÇÃO AO CÓDIGO DE PARANAGUÁ:
    #   Paranaguá: 0,80 fixo, H_ef = 8.760 h (calendário), todos os perfis.
    #   Este notebook: limite por número de berços (Quadro 17) e H_EF real.
    tab['N.º berços'] = 1   # padrão conservador; ajustar para berços conjugados (§3.2.1)
    tab['BOR_adm'] = tab['N.º berços'].apply(get_bor_adm_cont)
    tab['BOR_adm fonte'] = 'Quadro 17 (UNCTAD)'

    # --- Eq. 1b: Capacidade bruta em TEU/ano ---
    tab['C_cais bruta (TEU/ano)'] = (
        tab['BOR_adm'] * H_EF * tab['Lote médio (TEU)']
    ) / tab['Ta + a (h)']

    # Conversão para t/ano via fator TEU
    tab['C_cais bruta (t/ano)'] = tab['C_cais bruta (TEU/ano)'] * FATOR_TEU

    # --- Alocação por mix de berço ---
    tab['T_total_j (h)'] = (
        tab['Movimentação total (TEU)'] / tab['Lote médio (TEU)']
    ) * tab['Ta + a (h)']

    t_bercо = tab.groupby('Berço')['T_total_j (h)'].transform('sum')
    tab['T_total_berço (h)'] = t_bercо
    tab['f(j) — fração mix'] = tab['T_total_j (h)'] / tab['T_total_berço (h)']

    c_bruta_b = tab.groupby('Berço')['C_cais bruta (TEU/ano)'].transform('first')
    tab['C(j) alocada (TEU/ano)'] = c_bruta_b * tab['f(j) — fração mix']
    tab['C(j) alocada (t/ano)']   = tab['C(j) alocada (TEU/ano)'] * FATOR_TEU

    # --- Eq. 2a: BOR observado ---
    tab['BOR observado (%)'] = (
        tab['N.º atracações'] * tab['Ta — berth time (h)']
    ) / (tab['N.º berços'] * H_EF) * 100

    # --- Eq. 2b: BUR observado (em TEU) ---
    tab['BUR observado (%)'] = (
        tab['Movimentação total (TEU)'] / tab['C(j) alocada (TEU/ano)']
    ) * 100

    # --- Sinalização ---
    tab['BOR > BOR_adm?'] = tab['BOR observado (%)'] > (tab['BOR_adm'] * 100)

    print(f'  tabela_capacidades: {len(tab):,} linhas na Planilha 3')
    saturados = tab['BOR > BOR_adm?'].sum()
    if saturados > 0:
        print(f'  ATENÇÃO: {saturados} grupo(s) com BOR > BOR_adm.')
    return tab


tab_capacidades = tabela_capacidades_calc(tab_indicadores)
tab_capacidades.head(3)

In [ ]:
# --- Exporta Planilha 3 ---
tab_capacidades.to_excel('Planilha3_Capacidades_Cont.xlsx', index=False)
print('Planilha 3 exportada: Planilha3_Capacidades_Cont.xlsx')

In [ ]:
# ============================================================
# 6. SÍNTESE PARA O QUADRO 5 (contêineres)
# ============================================================
quadro5 = (
    tab_capacidades
    .groupby(['Complexo Portuário', 'Nome do Terminal', 'Berço'])
    .agg(
        C_cais_TEU_ano   = ('C(j) alocada (TEU/ano)', 'sum'),
        C_cais_t_ano     = ('C(j) alocada (t/ano)',   'sum'),
        Mov_TEU_ano      = ('Movimentação total (TEU)', 'sum'),
        BOR_obs_pct      = ('BOR observado (%)', 'mean'),
        BOR_adm_pct      = ('BOR_adm', 'mean'),
        BUR_obs_pct      = ('BUR observado (%)', 'mean'),
        N_atracacoes     = ('N.º atracações', 'sum'),
    )
    .reset_index()
)
quadro5.columns = [
    'Complexo Portuário', 'Terminal', 'Berço',
    'C_cais (TEU/ano)', 'C_cais (t/ano)',
    'Movimentação observada (TEU/ano)',
    'BOR obs. (%)', 'BOR adm. (%)', 'BUR obs. (%)', 'N.º atracações'
]
quadro5['BOR adm. (%)'] = quadro5['BOR adm. (%)'] * 100

quadro5.to_excel('Quadro5_Consolidacao_Sistemica_Cont.xlsx', index=False)
print('Quadro 5 (contêineres) exportado.')
quadro5

In [ ]:
# ============================================================
# 7. LOG DE DEPURAÇÃO
# ============================================================
bd2 = bd_depurada.merge(
    tab_indicadores[[
        'Apoio Concatenar',
        'L.Inf_Inop.Pré','L.Sup_Inop.Pré',
        'L.Inf_Produtiv.','L.Sup_Produtiv.',
        'L.Inf_Inop.Pós','L.Sup_Inop.Pós'
    ]],
    on='Apoio Concatenar', how='left'
)

def contar_retidos(grp, col, col_inf, col_sup):
    mask = (grp[col] >= grp[col_inf]) & (grp[col] <= grp[col_sup])
    return mask.sum()

log_rows = []
for chave, grp in bd2.groupby('Apoio Concatenar'):
    log_rows.append({
        'Apoio Concatenar': chave,
        'N total': len(grp),
        'N retidos Inop.Pré':   contar_retidos(grp, 'Inop.Pré',            'L.Inf_Inop.Pré',  'L.Sup_Inop.Pré'),
        'N retidos Produtiv.':  contar_retidos(grp, 'Produtividade (TEU/h)','L.Inf_Produtiv.', 'L.Sup_Produtiv.'),
        'N retidos Inop.Pós':   contar_retidos(grp, 'Inop.Pós',            'L.Inf_Inop.Pós',  'L.Sup_Inop.Pós'),
    })

log_df = pd.DataFrame(log_rows)
log_df.to_excel('Log_Depuracao_IQR_Cont.xlsx', index=False)
print(f'Log exportado: {len(log_df):,} grupos | Log_Depuracao_IQR_Cont.xlsx')

---
## Arquivos gerados

| Arquivo | Corresponde a | Uso no roteiro |
|---|---|---|
| `Planilha1_Base_Depurada_Cont.xlsx` | Planilha 1 (§11.2) | Base com replicatas tratadas e tempos calculados |
| `Planilha2_Indicadores_Operacionais_Cont.xlsx` | Planilha 2 (§11.2) | Indicadores depurados por IQR, em TEU |
| `Planilha3_Capacidades_Cont.xlsx` | Planilha 3 (§11.2) | C_cais (TEU/ano e t/ano), BOR, BUR |
| `Quadro5_Consolidacao_Sistemica_Cont.xlsx` | Quadro 5 (§9.2) | Template de consolidação sistêmica (contêineres) |
| `Log_Depuracao_IQR_Cont.xlsx` | Log (§11.3) | Rastreabilidade da depuração por IQR |

**Notas de calibração:**  
- `FATOR_TEU`: calibrar com dados do Estatístico ANTAQ do terminal (movimentação total TEU ÷ N.º atracações do grupo)  
- `BOR_ADM_PADRAO`: ajustar conforme número de berços do terminal e Quadro 17 do roteiro  
- `H_EF`: calcular pela Eq. 1c para o porto em análise antes de usar o valor padrão de 8.000 h  
- Para berços conjugados, aplicar o fator F_conj descrito no §3.2.1 do roteiro antes de rodar este notebook
